In [1]:
import os
from pathlib import Path
import sys

# PDF libraries
from pypdf import PdfReader

# Docling
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, RapidOcrOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

# Regex
import re

# OpenRouter
from openrouter import OpenRouter
from openrouter.operations import CreateEmbeddingsResponse

# env
from dotenv import load_dotenv

# Calculate token size
from pytokencalc import count_tokens

# Import from project files
sys.path.append('..')

from config import open_router_client
from config import supabase_client


c:\Users\alexi\studcess\api\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 0 - define the sample docs path

In [2]:
if (Path(os.getcwd()) != Path("c:/Users/alexi/studcess/api")):
    os.chdir("../../../api")

print(os.getcwd())

docs = Path("sample-data")
sample = docs / "exam1-2022-sol-1.pdf"

c:\Users\alexi\studcess\api


# 1 - extract document metadata

In [3]:
def extract_metadata(pdf_path: Path)->None:
    reader = PdfReader(pdf_path)
    metadata = reader.metadata
    num_pages = len(reader.pages)
    
    print(metadata)
    print(num_pages)
    
    # Build custom metadata
    info = {num_pages:num_pages}
    
    # Regexes for file class 
    EXAM_KEYWORDS = ["Exam"]
    
extract_metadata(sample)

{'/Producer': 'pdfTeX-1.40.20', '/Author': "Dr Pierre-André Mudry, Filière ISC - Haute Ecole d'Ingénierie, Sion", '/Title': 'Test intermédiaire', '/Subject': '', '/Creator': 'LaTeX with hyperref', '/Keywords': '101 - Fondements de la programmation 1, exercises', '/CreationDate': "D:20221109160320+01'00'", '/ModDate': "D:20221109160320+01'00'", '/Trapped': '/False', '/PTEX.Fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.20 (TeX Live 2019/Debian) kpathsea version 6.3.1'}
12


# 2 - convert document to markdown with docling

In [4]:
%%time

pipeline_options = PdfPipelineOptions(
    do_ocr=True,
    do_table_structure=True,
    generate_page_images=False,
    do_picture_description=False,
    ocr_options=RapidOcrOptions(),
    #picture_description_options=smolvlm_picture_description
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options),
    }
)

PAGE_BREAK_PLACEHOLDER = "<!-- page_break -->"

def process_document(pdf_path: Path,converter: DocumentConverter, n_pages=-1)->str:
    result = converter.convert(pdf_path)
    doc = result.document
    print(f"\nNumber of pages: {len(doc.pages)}\n")

    converted_dir = docs / "converted"
    converted_dir.mkdir(parents=True, exist_ok=True)
    
    doc.save_as_markdown(converted_dir / f"{pdf_path.stem}.md")
    doc.save_as_json(converted_dir / f"{pdf_path.stem}.json")
    
    markdown = doc.export_to_markdown(page_break_placeholder=PAGE_BREAK_PLACEHOLDER)
    
    if n_pages == -1:
        return markdown
    
    return PAGE_BREAK_PLACEHOLDER.join(markdown.split(PAGE_BREAK_PLACEHOLDER)[:n_pages])
    
doc = process_document(sample,converter,2)
print(doc)

Loading weights: 100%|██████████| 770/770 [00:00<00:00, 3436.89it/s]



Number of pages: 12

Prénom : . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

<!-- image -->

## TEST INTERMÉDIAIRE SOLUTION 101.1 - Programmation impérative

## Consigne :

Lisez attentivement la donnée et répondez de manière lisible aux questions. Vous avez droit pour cet examen à un aide-mémoire de 1 pages (1 feuille recto). Aucun moyen électronique n'est permis.

Un conseil : ne restez pas bloqués sur une question. Répondez tout d'abord aux questions avec lesquelles vous êtes à l'aise et revenez ensuite aux questions posant problème. Le barème indiqué est indicatif.

| Question              |   Points |   Bonus | Score   |
|-----------------------|----------|---------|---------|
| Short questions       |       10 |       2 |         |
| Loops analysis        |        8 |       0 |         |
| EBNF grammars         |        6 |       0 |         |
| Code comprehension    |        3 |       0 |         |
| Writing functions     |        7 |       1 |         |
| S

# 3 - Implement chunking 

We do a first pass by splitting by heading as this will be where we are most likely to get relevant information

In [5]:
HEADING_PATTERN = "\n#"

raw_chunks = doc.split(HEADING_PATTERN)

chunked_text = ""
for i,chunk in enumerate(raw_chunks):
    if chunk.startswith("#"):
        chunk = f"#{chunk}" # Put back the header
    chunked_text += f"<start_chunk_{i}>\n{chunk}<end_chunk_{i}>"

print(chunked_text)

<start_chunk_0>
Prénom : . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

<!-- image -->
<end_chunk_0><start_chunk_1>
## TEST INTERMÉDIAIRE SOLUTION 101.1 - Programmation impérative
<end_chunk_1><start_chunk_2>
## Consigne :

Lisez attentivement la donnée et répondez de manière lisible aux questions. Vous avez droit pour cet examen à un aide-mémoire de 1 pages (1 feuille recto). Aucun moyen électronique n'est permis.

Un conseil : ne restez pas bloqués sur une question. Répondez tout d'abord aux questions avec lesquelles vous êtes à l'aise et revenez ensuite aux questions posant problème. Le barème indiqué est indicatif.

| Question              |   Points |   Bonus | Score   |
|-----------------------|----------|---------|---------|
| Short questions       |       10 |       2 |         |
| Loops analysis        |        8 |       0 |         |
| EBNF grammars         |        6 |       0 |         |
| Code comprehension    |        3 |       0 |         |
| Writing 

# 4 - Embed chunks

## Golden rule for chunk sizes
We divide the length of text by 4 to get the approximate token count:
- Between 100 and 1000 => acceptable
- Less than 100 => too low, merge with another chunk or check for junk and discard
- More than 1000 => too big, split into smaller chunks needed

In [ ]:
class Chunk:
    def __init__(self,embedding,text):
        self.embedding = embedding
        self.text = text
        self.row = {
            "content": self.text,
            "embedding": self.embedding
        }
    
    def __repr__(self):
        return f"{self.row}"

def embed_chunks(raw_chunks: list[str], embedding_model: str) -> list[Chunk]:
    chunks: list[Chunk] = []

    for text in raw_chunks:
        tokens_count = len(text) // 4
        print(tokens_count)

        embedding = open_router_client.embeddings.generate(
            input=text,
            model=embedding_model,
        )

        print(text)
        chunks.append(Chunk(embedding, text))

    return chunks

chunks = embed_chunks(raw_chunks,"openai/text-embedding-3-small")

22
Prénom : . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

<!-- image -->



TypeError: float() argument must be a string or a real number, not 'CreateEmbeddingsResponseBody'

In [7]:
def store_chunks(chunks:list[Chunk]):
    for chunk in chunks:
        try:
            response = supabase_client.table("chunks").insert(chunk.row).execute()
            print(f"Successfully inserted {len(response.data)} chunks.")
        except Exception as e:
            print(f"Error inserting data: {e}")

store_chunks(chunks)

Error inserting data: Object of type CreateEmbeddingsResponseBody is not JSON serializable
Error inserting data: Object of type CreateEmbeddingsResponseBody is not JSON serializable
Error inserting data: Object of type CreateEmbeddingsResponseBody is not JSON serializable
Error inserting data: Object of type CreateEmbeddingsResponseBody is not JSON serializable
